# Phishing Email Detection with Logistic Regression

This notebook walks through the complete process of building a machine learning model to classify emails as either 'Phishing' or 'Safe'.

The process includes:
1. Loading and combining multiple datasets.
2. Cleaning and balancing the data to prevent bias.
3. Converting email text into numerical features (Vectorization).
4. Training a Logistic Regression model.
5. Evaluating the model's accuracy.
6. Testing the model with a new, unseen email.

## 1. Import Libraries

First, we import all the necessary libraries for data manipulation, feature extraction, and machine learning.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 2. Load and Combine Datasets

We will load our two separate CSV files. They will be combined into a single master DataFrame for processing.

In [ ]:
# List of file paths for all your datasets
file_paths = [
    'data/Phishing_validation_emails(1).csv',  # Your original file
    'Phishing_Email.csv'                       # Your new file
]

try:
    # Read each file and store it in a list of DataFrames
    list_of_dfs = [pd.read_csv(file) for file in file_paths]
    
    # Combine all DataFrames in the list into one
    df = pd.concat(list_of_dfs, ignore_index=True)
    
    print(f"--- Loaded and combined {len(file_paths)} files successfully ---")
    print(f"Total emails before cleaning: {len(df)}")

except FileNotFoundError as e:
    print(f"Error: The file was not found. Please check the path: {e.filename}")
except Exception as e:
    print(f"An error occurred: {e}")

## 3. Clean and Balance the Data

Data cleaning is crucial. We will remove any empty or duplicate emails. Then, to prevent the model from being biased, we will balance the dataset so there are an equal number of phishing and safe emails for the model to learn from.

In [ ]:
# Remove any rows with missing text or types
df.dropna(subset=['Email Text', 'Email Type'], inplace=True)

# Remove duplicate emails based on the 'Email Text' column
df.drop_duplicates(subset=['Email Text'], inplace=True)
print(f"Total unique emails after cleaning: {len(df)}\n")

# --- Balance the Dataset ---
safe_email = df[df["Email Type"] == 'Safe Email']
phishing_email = df[df["Email Type"] == 'Phishing Email']

print(f"Original counts: {len(safe_email)} safe emails, {len(phishing_email)} phishing emails.")

# Balance by downsampling the larger class
if len(safe_email) > len(phishing_email):
    safe_email = safe_email.sample(n=len(phishing_email), random_state=42)
elif len(phishing_email) > len(safe_email):
    phishing_email = phishing_email.sample(n=len(safe_email), random_state=42)

# Combine the balanced sets into a final dataframe
balanced_data = pd.concat([safe_email, phishing_email], ignore_index=True)
print(f"--- Dataset is now balanced. Total samples for training: {len(balanced_data)} ---")

## 4. Prepare Data for Modeling (Vectorization)

A machine learning model can't read text. We need to convert the email text into numbers. We'll use `CountVectorizer` to do this, which turns each email into a vector of word counts.

In [ ]:
# Initialize the vectorizer
vectorizer = CountVectorizer(stop_words='english')

# Create the feature matrix (X) from the email text
X = vectorizer.fit_transform(balanced_data['Email Text'])

# Create the target labels (y), where 1 = Phishing and 0 = Safe
y = balanced_data['Email Type'].apply(lambda x: 1 if x == 'Phishing Email' else 0)

print("Data has been vectorized successfully.")
print("Shape of the feature matrix (X):", X.shape)

## 5. Split Data for Training and Testing

We'll split our data into a training set (80%) and a testing set (20%). The model learns from the training set and is then evaluated on the unseen testing set to see how well it performs.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

## 6. Train the Logistic Regression Model

Now we train our model using the `fit` method on the training data.

In [ ]:
# Initialize the model
model = LogisticRegression(max_iter=1000)

# Train the model
print("--- Training the model... ---")
model.fit(X_train, y_train)
print("--- Model training complete! ---")

## 7. Evaluate the Model

Let's see how well our model did by predicting the labels for our unseen test set and comparing them to the actual labels.

In [ ]:
# Make predictions on the test data
y_pred = model.predict(X_test)

# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on Test Data: {accuracy:.4f}")

## 8. Test the Model with a New Email

This is the final step where we can use our trained model to classify a brand new email.

In [ ]:
# You can change the text inside the triple quotes to test any email
new_email = """
URGENT: Your account has been suspended due to suspicious activity. Click here to verify your details immediately to avoid permanent closure.
"""

# Use the SAME vectorizer to transform the new email
new_email_transformed = vectorizer.transform([new_email])

# Make a prediction
prediction = model.predict(new_email_transformed)
prediction_proba = model.predict_proba(new_email_transformed)

# Print the result
print(f"Email Text: \n{new_email}")
if prediction[0] == 1:
    print(f"\nPrediction: This is a Phishing Email 🎣")
    print(f"Confidence: {prediction_proba[0][1]:.2%}")
else:
    print(f"\nPrediction: This is a Safe Email ✅")
    print(f"Confidence: {prediction_proba[0][0]:.2%}")